# WTI Crude Oil — Adaptive Agent Training (Notebook 5 of 7)

> **Part 5 of 7.** Builds on the stateless backtest in [`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb).

Every method in Notebook 4 was **stateless** — configured once and run.  
This notebook introduces an agent that is different: it has a **training phase**.

The paradigm shift: instead of configuring a model, we onboard an analyst.  
We give the analyst historical performance data to study, let it draw conclusions
and update its own strategy, then put it on live forecasting duty in Notebook 6.

The training paradigm is **curriculum learning** — not time-travel simulation.  
We prepare structured learning material (backtest reports, pre-cached news context)
and hand it to the agent for reflection. The agent decides what to record, based
on the evidence governance rules in its `meta-learning` skill.

**Two training variants are produced:**

| Variant | Strategy dir | Training material |
|---------|-------------|------------------|
| Stats-only | `wti-strategy-stats/` | Activity 1 (exploration) + Activity 2a (backtest report) |
| News-grounded | `wti-strategy-news/` | Activity 2b (same report + weekly news context) |

---
## 0. Setup

In [4]:
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown
from IPython.display import display as ipy_display  # noqa: F401

from aieng.forecasting.evaluation.backtest import BacktestResult
from aieng.forecasting.methods.agentic import (
    build_adk_agent,
    build_curriculum_prompt,
    format_backtest_report,
    load_context_documents,
)
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig
from energy_oil_forecasting.adaptive_agent import (
    build_wti_adaptive_config,
)
from energy_oil_forecasting.adaptive_agent.curriculum.snapshot_utils import (
    restore_state,
    snapshot_state,
)
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path('.')
_SKILLS_ROOT = _NB_DIR / 'adaptive_agent' / 'skills'
_CURRICULUM_DIR = _NB_DIR / 'adaptive_agent' / 'curriculum'
_CONTEXT_DIR = _CURRICULUM_DIR / 'context'

# Clean seed — never modified by training activities.
SEED_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy'
# One independent variant per training activity.
ACT1_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-act1'   # Act 1: self-directed
STATS_STRATEGY_DIR = _SKILLS_ROOT / 'wti-strategy-stats'  # Act 2a: stats curriculum
NEWS_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-news'   # Act 2b: news curriculum

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = 'gemini-3.5-flash'

# ── Run guards ────────────────────────────────────────────────────────────────
# Expensive activities default to False (outputs committed after first run).
# Set True only when you want to regenerate outputs from scratch.
# Each activity is INDEPENDENT: re-running one does not affect the others.
RUN_ACTIVITY_1  = True   # Act 1: agent-initiated code-execution exploration
RUN_ACTIVITY_2A = True   # Act 2a: statistics-only curriculum delivery
RUN_ACTIVITY_2B = True   # Act 2b: news-grounded curriculum delivery

# ── Data service ──────────────────────────────────────────────────────────────
data_service = build_wti_service()
print('Setup complete.')

Setup complete.


---
## 1. Three Independent Variants — Seeded from Clean State

Each training activity writes to its **own isolated strategy directory**,  
seeded from the canonical clean starting point (`wti-strategy/`).  
This means:

- Activities are independent — re-running one does not affect the others.
- Each variant starts from the same prior, so differences in their final  
  strategy state are attributable to the training experience, not to ordering.
- The clean seed (`wti-strategy/`) is never modified by any training activity.

| Run guard | Strategy directory | Training experience |
|---|---|---|
| `RUN_ACTIVITY_1` | `wti-strategy-act1/` | Self-directed code exploration |
| `RUN_ACTIVITY_2A` | `wti-strategy-stats/` | Statistics-only curriculum |
| `RUN_ACTIVITY_2B` | `wti-strategy-news/` | News-grounded curriculum |

The cell below re-seeds each variant from the clean seed before any agent  
runs. It is safe to re-run at any time to reset all variants to the initial state.

In [5]:
import shutil

def _reseed(variant_dir: Path) -> None:
    """Copy skill_state.yaml from the clean seed into variant_dir and re-render SKILL.md."""
    from aieng.forecasting.methods.agentic.adaptive_skill import AdaptiveSkillStore
    from energy_oil_forecasting.adaptive_agent.skill_state import WtiStrategyState
    variant_dir.mkdir(exist_ok=True)
    shutil.copy2(SEED_STRATEGY_DIR / 'skill_state.yaml', variant_dir / 'skill_state.yaml')
    store = AdaptiveSkillStore(skill_dir=variant_dir, state_type=WtiStrategyState)
    state = store.load()
    (variant_dir / 'SKILL.md').write_text(state.build_markdown(skill_name=variant_dir.name))
    print(f'  Seeded {variant_dir.name}/')

print('Re-seeding all variant strategy directories from clean seed...')
_reseed(ACT1_STRATEGY_DIR)
_reseed(STATS_STRATEGY_DIR)
_reseed(NEWS_STRATEGY_DIR)
print('Done. All three variants are at the clean initial state.')
print()
print('Clean initial SKILL.md:')
print('─' * 60)
print((SEED_STRATEGY_DIR / 'SKILL.md').read_text())

Re-seeding all variant strategy directories from clean seed...
  Seeded wti-strategy-act1/
  Seeded wti-strategy-stats/
  Seeded wti-strategy-news/
Done. All three variants are at the clean initial state.

Clean initial SKILL.md:
────────────────────────────────────────────────────────────
---
name: wti-strategy
description: >-
  The adaptive WTI analyst's current forecasting strategy. Load this at the
  start of every prediction task. This file is generated — edit the state
  through the mutation tools, not by hand.
---

# WTI Forecasting Strategy

## Approach

Produce calibrated probabilistic forecasts by combining two evidence streams:
statistical analysis of recent price history and web-grounded news context.

At short horizons (5 bd), momentum and recent trend dominate. Trust the trend
projection output unless there is a strong near-term catalyst visible in news
context (e.g. an imminent OPEC+ meeting or scheduled inventory release).

At medium horizons (10 bd), OPEC+ meeting sche

---
## 2. Activity 1 — Self-Directed Exploration (`wti-strategy-act1`)

We give the agent access to historical WTI price data via code execution  
and ask an open-ended analytical question. The agent decides what to  
compute, draws its own conclusions, and decides whether findings meet  
the evidence threshold in `meta-learning`.

**This is the 'analyst left alone with data' variant.** No pre-packaged  
report, no curated news — the agent's strategy updates (if any) are driven  
entirely by its own analytical choices.

Writes to: `wti-strategy-act1/`

> **Run guard:** `RUN_ACTIVITY_1 = False` by default — outputs are committed  
> so the notebook runs reproducibly without real API calls.

In [6]:
_ACTIVITY_1_PROMPT = (
    'You have access to historical WTI crude oil price data via run_code. '
    'Please do the following:\n\n'
    '1. Fetch the daily WTI close price series for the full year 2025 using '
    'yfinance (ticker: CL=F).\n'
    '2. Compute 21-day rolling realized volatility. Classify each day into a '
    'vol regime: low (<15% annualized), medium (15-30%), elevated (30-50%), '
    'or extreme (>50%).\n'
    '3. Simulate the errors a simple trend-projection forecaster would make '
    'at 5, 10, and 21 business-day horizons during each regime. Approximate '
    'this using the historical return distribution within each regime window.\n'
    '4. Summarize: in which regimes and at which horizons does trend-projection '
    'tend to produce the largest errors? Is there a directional bias?\n\n'
    'Based on your analysis, decide whether any findings meet the evidence '
    'threshold in your meta-learning skill. If they do, record them. '
    'If not, explain what additional evidence you would need.'
)

if RUN_ACTIVITY_1:
    config = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=ACT1_STRATEGY_DIR
    )
    agent = build_adk_agent(config)
    runner = AdkTextRunner(
        agent,
        config=AdkTextRunnerConfig(
            app_name='wti_training_act1',
            enable_langfuse_tracing=True,
            langfuse_tags=['energy-oil', 'adaptive-agent', 'activity-1'],
            langfuse_trace_name='wti-adaptive-activity-1',
        ),
    )
    print('Running Activity 1 (code execution + reflection)...')
    print('This may take several minutes.\n')
    reply_act1 = await runner.run_text_async(_ACTIVITY_1_PROMPT)
    (_CURRICULUM_DIR / 'activity1_response.txt').write_text(
        reply_act1, encoding='utf-8'
    )
    print(reply_act1)
else:
    _f = _CURRICULUM_DIR / 'activity1_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 1 output not yet committed. '
              'Set RUN_ACTIVITY_1 = True and re-run.]')

Running Activity 1 (code execution + reflection)...
This may take several minutes.

---------------------------------------------------------------------------TypeError                                 Traceback (most recent call last)Cell In[1], line 97
     94 results_df = pd.DataFrame(results)
     96 # Let's see some summary statistics grouped by window, regime, and horizon
---> 97 summary = results_df.groupby(["window", "regime", "horizon"]).agg(
     98     n=("error", "count"),
     99     me=("error", "mean"),
    100     mae=("error", "mean", lambda x: np.mean(np.abs(x))),
    101     rmse=("error", "mean", lambda x: np.sqrt(np.mean(x**2))),
    102     std_err=("error", "std"),
    103     min_err=("error", "min"),
    104     max_err=("error", "max")
    105 ).reset_index()
    107 print("--- TREND PROJECTION ERROR SUMMARY ---")
    108 print(summary.to_string(index=False))
File /usr/local/lib/python3.13/site-packages/pandas/core/groupby/generic.py:1422, in DataFrameGroupBy.a

Root node wti_adaptive_analyst was cancelled.


In [7]:
print('wti-strategy-act1/SKILL.md after Activity 1:')
print('─' * 60)
print((ACT1_STRATEGY_DIR / 'SKILL.md').read_text())

wti-strategy-act1/SKILL.md after Activity 1:
────────────────────────────────────────────────────────────
---
name: wti-strategy-act1
description: >-
  The adaptive WTI analyst's current forecasting strategy. Load this at the
  start of every prediction task. This file is generated — edit the state
  through the mutation tools, not by hand.
---

# WTI Forecasting Strategy

## Approach

Produce calibrated probabilistic forecasts by combining two evidence streams:
statistical analysis of recent price history and web-grounded news context.

At short horizons (5 bd), momentum and recent trend dominate. Trust the trend
projection output unless there is a strong near-term catalyst visible in news
context (e.g. an imminent OPEC+ meeting or scheduled inventory release).

At medium horizons (10 bd), OPEC+ meeting schedules and US inventory release
dates matter. Check for scheduled events in the news context before finalising
the forecast.

At long horizons (21 bd), macro demand and geopolitical

---
## 3. Activity 2a — Statistics-Only Curriculum (`wti-strategy-stats`)

We compile the 2025 backtest results from Notebook 4 into a structured  
report (per-horizon coverage, bias, MAE, interval width, regime breakdown)  
and send it to the agent as a curriculum document. No news context is  
provided — the agent's updates are driven by quantitative evidence alone.

**This is the 'pure statistics' variant.** Comparing it to Activity 1  
isolates the effect of structured backtest feedback vs. self-directed exploration.

Writes to: `wti-strategy-stats/`

> **Run guard:** `RUN_ACTIVITY_2A = False` by default.

In [8]:
# ── Load 2025 backtest results saved by NB04 ────────────────────────────────
_backtest_jsons = sorted(_CURRICULUM_DIR.glob('backtest_*.json'))
if not _backtest_jsons:
    raise FileNotFoundError(
        'No backtest result files found in adaptive_agent/curriculum/. '
        'Run 04_systematic_backtest_eval.ipynb first.'
    )

backtest_results = {}
for f in _backtest_jsons:
    name = f.stem.removeprefix('backtest_')
    backtest_results[name] = BacktestResult.model_validate_json(f.read_text())

print(f'Loaded {len(backtest_results)} backtest result(s):')
for name, r in backtest_results.items():
    print(f'  {name}: {len(r.predictions)} predictions, '
          f'mean CRPS = {r.mean_crps:.4f}')

Loaded 3 backtest result(s):
  AutoARIMA: 145 predictions, mean CRPS = 2.4721
  Naive (Last Value): 145 predictions, mean CRPS = 2.9299
  Prophet: 95 predictions, mean CRPS = 10.3661


In [9]:
# ── Build actuals dict (needed by format_backtest_report) ───────────────────
# get_series returns a DataFrame with 'timestamp' and 'value' columns.
# We use as_of=datetime.now() so all 2025 actuals are available (no cutoff).
from datetime import datetime  # noqa: PLC0415

_best_name = min(backtest_results, key=lambda n: backtest_results[n].mean_crps)
_best_result = backtest_results[_best_name]
print(f"Using '{_best_name}' (mean CRPS = {_best_result.mean_crps:.4f}) "
      'as curriculum basis.')

_full_series = data_service.get_series(WTI_SERIES_ID, as_of=datetime.now())

actuals: dict[tuple[str, int], float] = {}
for pred in _best_result.predictions:
    horizon = (pred.forecast_date - pred.as_of).days
    target_ts = pd.Timestamp(pred.forecast_date)
    match = _full_series[pd.to_datetime(_full_series['timestamp']) == target_ts]
    if not match.empty:
        actuals[(str(pred.as_of.date()), horizon)] = float(match['value'].iloc[0])

print(f'Resolved {len(actuals)} actuals for '
      f'{len(_best_result.predictions)} predictions.')

Using 'AutoARIMA' (mean CRPS = 2.4721) as curriculum basis.
Resolved 145 actuals for 145 predictions.


In [10]:
# ── Format and display the backtest report ───────────────────────────────────
# baseline_result provides a naive comparison row; price_series enables
# per-vol-regime breakdowns.  Both are optional — omit if not available.
_naive_result = backtest_results.get('Naive (Last Value)')

report = format_backtest_report(
    result=_best_result,
    actuals=actuals,
    title=f'2025 WTI Backtest — {_best_name}',
    training_start=date(2025, 1, 1),
    training_end=date(2025, 12, 31),
    baseline_result=_naive_result,
    price_series=_full_series,
)
ipy_display(Markdown(report))

# 2025 WTI Backtest — AutoARIMA

**Predictor:** darts_autoarima  
**Origins included:** 51  
**Mean CRPS (all horizons):** 2.4721

## Relative skill vs. naive baseline

Baseline predictor: **last_value_naive**  
Baseline mean CRPS: 2.9299  
This predictor mean CRPS: 2.4721

| Horizon | This MAE | Baseline MAE | Skill (lower is better) |
|---------|----------|--------------|-------------------------|
| 7d | 2.40 | 2.34 | 2.40 vs 2.34 (worse by 0.06) |
| 14d | 2.80 | 2.81 | 2.80 vs 2.81 (better by 0.00) |
| 29d | 3.71 | 3.59 | 3.71 vs 3.59 (worse by 0.12) |

---

## Horizon: 7 days

| Metric | Value |
|--------|-------|
| Predictions resolved | 47 |
| 80% CI coverage | 91.5% (target 80%) |
| Mean bias (forecast − actual) | +0.44 (over-forecasting) |
| Mean absolute error | 2.40 |
| Average 80% CI width | 10.94 |
| Width needed for 80% coverage | ±3.65 (current half-width: ±5.47) |

> **Coverage 91.5% is above target** — intervals may be overly conservative at this horizon.

**Regime breakdown:**

| Vol regime | N | Coverage | MAE | Mean bias |
|-----------|---|----------|-----|-----------|
| medium | 34 | 94.1% | 2.27 | +0.14 |
| elevated | 11 | 81.8% | 3.01 | +1.68 |
| extreme | 2 | 100.0% | 1.35 | -1.35 |

## Horizon: 14 days

| Metric | Value |
|--------|-------|
| Predictions resolved | 47 |
| 80% CI coverage | 93.6% (target 80%) |
| Mean bias (forecast − actual) | +0.86 (over-forecasting) |
| Mean absolute error | 2.80 |
| Average 80% CI width | 15.55 |
| Width needed for 80% coverage | ±4.10 (current half-width: ±7.78) |

> **Coverage 93.6% is above target** — intervals may be overly conservative at this horizon.

**Regime breakdown:**

| Vol regime | N | Coverage | MAE | Mean bias |
|-----------|---|----------|-----|-----------|
| medium | 34 | 97.1% | 2.70 | +0.75 |
| elevated | 11 | 81.8% | 3.41 | +1.58 |
| extreme | 2 | 100.0% | 1.21 | -1.21 |

## Horizon: 29 days

| Metric | Value |
|--------|-------|
| Predictions resolved | 51 |
| 80% CI coverage | 98.0% (target 80%) |
| Mean bias (forecast − actual) | +1.49 (over-forecasting) |
| Mean absolute error | 3.71 |
| Average 80% CI width | 22.31 |
| Width needed for 80% coverage | ±5.36 (current half-width: ±11.15) |

> **Coverage 98.0% is above target** — intervals may be overly conservative at this horizon.

**Regime breakdown:**

| Vol regime | N | Coverage | MAE | Mean bias |
|-----------|---|----------|-----|-----------|
| medium | 37 | 100.0% | 3.31 | +1.96 |
| elevated | 12 | 91.7% | 5.08 | +0.49 |
| extreme | 2 | 100.0% | 2.80 | -1.19 |

---

## Cross-horizon pattern summary

- h=7d: coverage 91.5%, MAE 2.40, bias +0.44 (over), CI width 10.94 (needed 7.31)
- h=14d: coverage 93.6%, MAE 2.80, bias +0.86 (over), CI width 15.55 (needed 8.19)
- h=29d: coverage 98.0%, MAE 3.71, bias +1.49 (over), CI width 22.31 (needed 10.72)

Coverage **improves** across horizons. MAE **increases** with horizon. Bias is **consistent in direction** across all horizons (+), suggesting a structural over/under-forecast rather than a horizon-specific issue.


In [11]:
_PREAMBLE_2A = (
    'You are reviewing the 2025 WTI forecasting performance of the strongest '
    'stateless predictor from a systematic backtest. Study the per-horizon '
    'coverage and error statistics. Identify systematic patterns — particularly '
    'where coverage deviates from the 80% target or where MAE is unexpectedly '
    'large. Decide whether any findings meet the evidence threshold in your '
    'meta-learning skill, and if so, record them using the appropriate tools.'
)

prompt_2a = build_curriculum_prompt(
    report=report,
    context_documents=[],
    as_of='2025-12-31',
    preamble=_PREAMBLE_2A,
)

if RUN_ACTIVITY_2A:
    config_2a = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=STATS_STRATEGY_DIR
    )
    agent_2a = build_adk_agent(config_2a)
    runner_2a = AdkTextRunner(
        agent_2a,
        config=AdkTextRunnerConfig(
            app_name='wti_training_2a',
            enable_langfuse_tracing=True,
            langfuse_tags=['energy-oil', 'adaptive-agent', 'activity-2a', 'stats-only'],
            langfuse_trace_name='wti-adaptive-activity-2a',
        ),
    )
    print('Sending statistics-only curriculum...')
    reply_2a = await runner_2a.run_text_async(prompt_2a)
    (_CURRICULUM_DIR / 'activity2a_response.txt').write_text(
        reply_2a, encoding='utf-8'
    )
    print(reply_2a)
else:
    _f = _CURRICULUM_DIR / 'activity2a_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 2a output not yet committed. '
              'Set RUN_ACTIVITY_2A = True and re-run.]')

Sending statistics-only curriculum...
I have completed a thorough, systematic review of the 2025 WTI backtest results for the `darts_autoarima` predictor across 51 historical origins. 

Consistent with the **temporal and meta-learning disciplines** outlined in our strategy, I have analyzed the per-horizon coverage and error statistics, identified four major systematic patterns in our forecasting behavior, and recorded these findings in persistent memory.

Below is the detailed analytical breakdown of the backtest performance, followed by the specific structural actions taken to update our forecasting strategy.

---

# 2025 WTI Backtest Performance Report

## 1. Key Performance Metrics across Horizons

| Horizon | N | 80% CI Coverage | Mean Bias (F - A) | This MAE | Baseline MAE | Avg CI Width | Needed CI Width |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **7 days** | 47 | **91.5%** | +0.44 | 2.40 | 2.34 | 10.94 | ±3.65 (7.30 total) |
| **14 days** | 47 | **93.6%

Root node wti_adaptive_analyst was cancelled.


In [12]:
print('wti-strategy-stats/SKILL.md after Activity 2a:')
print('─' * 60)
print((STATS_STRATEGY_DIR / 'SKILL.md').read_text())

wti-strategy-stats/SKILL.md after Activity 2a:
────────────────────────────────────────────────────────────
---
name: wti-strategy-stats
description: >-
  The adaptive WTI analyst's current forecasting strategy. Load this at the
  start of every prediction task. This file is generated — edit the state
  through the mutation tools, not by hand.
---

# WTI Forecasting Strategy

## Approach

Produce calibrated probabilistic forecasts by combining two evidence streams:
statistical analysis of recent price history and web-grounded news context.

At short horizons (5 bd), momentum and recent trend dominate. Trust the trend
projection output unless there is a strong near-term catalyst visible in news
context (e.g. an imminent OPEC+ meeting or scheduled inventory release).

At medium horizons (10 bd), OPEC+ meeting schedules and US inventory release
dates matter. Check for scheduled events in the news context before finalising
the forecast.

At long horizons (21 bd), macro demand and geopoliti

---
## 4. Activity 2b — News-Grounded Curriculum (`wti-strategy-news`)

Same backtest report as Activity 2a, now augmented with pre-cached weekly  
news summaries from 2025. Each summary was generated with strict temporal  
cutoff enforcement, containing only information publicly available on that date.

**This is the 'statistics + market context' variant.** Comparing it to  
Activity 2a isolates the marginal effect of news grounding on top of  
quantitative feedback alone.

Writes to: `wti-strategy-news/`

> **Run guard:** `RUN_ACTIVITY_2B = False` by default.

In [13]:
# ── Representative news dates — one per month across 2025 ───────────────────
# Selected to cover OPEC+ meeting windows and seasonal demand inflection points.
_CURRICULUM_NEWS_DATES = [
    '2025-01-06',  # start of year
    '2025-02-03',  # pre-OPEC+ ministerial
    '2025-03-03',  # OPEC+ output decision period
    '2025-04-07',  # post-OPEC+ adjustment
    '2025-05-05',  # spring demand season
    '2025-06-09',  # OPEC+ June meeting
    '2025-07-07',  # summer demand peak
    '2025-08-04',  # late-summer
    '2025-09-08',  # OPEC+ September review
    '2025-10-06',  # Q4 demand build
    '2025-11-03',  # OPEC+ November decisions
    '2025-12-08',  # year-end
]

context_docs = load_context_documents(_CONTEXT_DIR, _CURRICULUM_NEWS_DATES)
print(f'Loaded {len(context_docs)} context documents:')
for d, content in context_docs:
    print(f'  {d}: {len(content):,} chars')

Loaded 12 context documents:
  2025-01-06: 3,877 chars
  2025-02-03: 3,948 chars
  2025-03-03: 3,827 chars
  2025-04-07: 4,235 chars
  2025-05-05: 3,946 chars
  2025-06-09: 4,224 chars
  2025-07-07: 3,697 chars
  2025-08-04: 3,630 chars
  2025-09-08: 1,510 chars
  2025-10-06: 3,826 chars
  2025-11-03: 4,533 chars
  2025-12-08: 4,725 chars


In [14]:
_PREAMBLE_2B = (
    'You are reviewing 2025 WTI forecasting performance alongside weekly market '
    'context summaries from the same period. The backtest report shows '
    'statistical patterns; the context summaries show what information was '
    'available at key dates. Study both together: does the news context help '
    'explain the error patterns? Identify systematic patterns and decide '
    'whether they meet the evidence threshold in your meta-learning skill. '
    'If so, record them using the appropriate tools.'
)

prompt_2b = build_curriculum_prompt(
    report=report,
    context_documents=context_docs,
    as_of='2025-12-31',
    preamble=_PREAMBLE_2B,
)
print(f'Curriculum prompt: {len(prompt_2b):,} chars '
      f'({len(context_docs)} context documents)')

Curriculum prompt: 50,753 chars (12 context documents)


In [15]:
if RUN_ACTIVITY_2B:
    config_2b = build_wti_adaptive_config(
        model=AGENT_MODEL, strategy_dir=NEWS_STRATEGY_DIR
    )
    agent_2b = build_adk_agent(config_2b)
    runner_2b = AdkTextRunner(
        agent_2b,
        config=AdkTextRunnerConfig(
            app_name='wti_training_2b',
            enable_langfuse_tracing=True,
            langfuse_tags=['energy-oil', 'adaptive-agent', 'activity-2b', 'news-grounded'],
            langfuse_trace_name='wti-adaptive-activity-2b',
        ),
    )
    print('Sending news-grounded curriculum...')
    reply_2b = await runner_2b.run_text_async(prompt_2b)
    (_CURRICULUM_DIR / 'activity2b_response.txt').write_text(
        reply_2b, encoding='utf-8'
    )
    print(reply_2b)
else:
    _f = _CURRICULUM_DIR / 'activity2b_response.txt'
    if _f.exists():
        print(_f.read_text())
    else:
        print('[Activity 2b output not yet committed. '
              'Set RUN_ACTIVITY_2B = True and re-run.]')

Sending news-grounded curriculum...
An analysis of the 2025 WTI crude oil backtest performance of the `darts_autoarima` model has been conducted, synthesizing the statistical performance metrics with the weekly market context from the same period. 

Two highly systematic forecasting patterns have been identified that meet the evidence threshold specified in the `meta-learning` skill. The appropriate strategy mutation tools have been invoked to record these observations and open two formal hypotheses for active tracking.

---

# 2025 WTI Backtest Analysis & Market Context Synthesis

## 1. Summary of Statistical Findings
Over the 51 historical origins in 2025, the AutoARIMA model achieved a Mean CRPS of **2.4721**, which represents a significant improvement over the naive baseline (**2.9299**). However, a closer look at the horizon-specific metrics reveals two major, persistent errors:
1. **Severe Over-Forecasting Bias:** The model exhibits a positive bias across all horizons, which esca

Root node wti_adaptive_analyst was cancelled.


In [16]:
print('wti-strategy-news/SKILL.md after Activity 2b:')
print('─' * 60)
print((NEWS_STRATEGY_DIR / 'SKILL.md').read_text())

wti-strategy-news/SKILL.md after Activity 2b:
────────────────────────────────────────────────────────────
---
name: wti-strategy-news
description: >-
  The adaptive WTI analyst's current forecasting strategy. Load this at the
  start of every prediction task. This file is generated — edit the state
  through the mutation tools, not by hand.
---

# WTI Forecasting Strategy

## Approach

Produce calibrated probabilistic forecasts by combining two evidence streams:
statistical analysis of recent price history and web-grounded news context.

At short horizons (5 bd), momentum and recent trend dominate. Trust the trend
projection output unless there is a strong near-term catalyst visible in news
context (e.g. an imminent OPEC+ meeting or scheduled inventory release).

At medium horizons (10 bd), OPEC+ meeting schedules and US inventory release
dates matter. Check for scheduled events in the news context before finalising
the forecast.

At long horizons (21 bd), macro demand and geopolitica

---
## 5. Side-by-Side Comparison

Three independent training runs from the same clean starting point.  
What did each variant learn, and how do the resulting strategies differ?

| Variant | Training input | Key question |
|---|---|---|
| `wti-strategy-act1` | Self-directed code exploration | What does the agent notice on its own? |
| `wti-strategy-stats` | Structured backtest report (stats only) | How does quantitative feedback shape priors? |
| `wti-strategy-news` | Backtest report + weekly news context | Does market context shift the strategy further? |

All three variants are evaluated in Notebook 6 against the same 2026 eval spec,  
alongside the untrained agent and the stateless methods from Notebook 4.

In [17]:
def _load_yaml_state(strategy_dir: Path) -> dict:
    return yaml.safe_load((strategy_dir / 'skill_state.yaml').read_text())

act1_state  = _load_yaml_state(ACT1_STRATEGY_DIR)
stats_state = _load_yaml_state(STATS_STRATEGY_DIR)
news_state  = _load_yaml_state(NEWS_STRATEGY_DIR)

rows = []
for label, state in [
    ('Act 1 — self-directed  (wti-strategy-act1)', act1_state),
    ('Act 2a — stats only    (wti-strategy-stats)', stats_state),
    ('Act 2b — news-grounded (wti-strategy-news)', news_state),
]:
    rows.append({
        'Variant': label,
        'Observations':            len(state.get('observations', [])),
        'Hypotheses':              len(state.get('hypotheses', [])),
        'Calibration corrections': len(state.get('calibration_corrections', [])),
    })

df_comparison = pd.DataFrame(rows).set_index('Variant')
print('Training outcomes — knowledge accumulated per variant:')
print(df_comparison.to_string())

Training outcomes — knowledge accumulated per variant:
                                             Observations  Hypotheses  Calibration corrections
Variant                                                                                       
Act 1 — self-directed  (wti-strategy-act1)              2           1                        0
Act 2a — stats only    (wti-strategy-stats)             9           2                        0
Act 2b — news-grounded (wti-strategy-news)              4           2                        0


In [18]:
for label, d in [
    ('wti-strategy-act1  (Activity 1)', ACT1_STRATEGY_DIR),
    ('wti-strategy-stats (Activity 2a)', STATS_STRATEGY_DIR),
    ('wti-strategy-news  (Activity 2b)', NEWS_STRATEGY_DIR),
]:
    print(f'\n── {label} ──')
    print((d / 'SKILL.md').read_text())


── wti-strategy-act1  (Activity 1) ──
---
name: wti-strategy-act1
description: >-
  The adaptive WTI analyst's current forecasting strategy. Load this at the
  start of every prediction task. This file is generated — edit the state
  through the mutation tools, not by hand.
---

# WTI Forecasting Strategy

## Approach

Produce calibrated probabilistic forecasts by combining two evidence streams:
statistical analysis of recent price history and web-grounded news context.

At short horizons (5 bd), momentum and recent trend dominate. Trust the trend
projection output unless there is a strong near-term catalyst visible in news
context (e.g. an imminent OPEC+ meeting or scheduled inventory release).

At medium horizons (10 bd), OPEC+ meeting schedules and US inventory release
dates matter. Check for scheduled events in the news context before finalising
the forecast.

At long horizons (21 bd), macro demand and geopolitical risk dominate. The
statistical signal loses explanatory power at t

---
## 6. Reset

Re-run the seed cell in Section 1 at any time to reset all three variant  
directories to the clean initial state. This lets you re-run any activity  
from scratch without re-running Notebook 4.

Activities are independent — resetting one does not affect the others.  
If you want to reset only one variant, call `_reseed(ACT1_STRATEGY_DIR)`,  
`_reseed(STATS_STRATEGY_DIR)`, or `_reseed(NEWS_STRATEGY_DIR)` individually.

In [19]:
# ── Re-seed all variants to the clean initial state ─────────────────────────
# Re-runs the same seed logic as Section 1. Safe to run at any time.
# Uncomment and run to reset:
#
# _reseed(ACT1_STRATEGY_DIR)
# _reseed(STATS_STRATEGY_DIR)
# _reseed(NEWS_STRATEGY_DIR)
# print('All three variants reset to clean initial state.')